# Classification Metrics · The delivery status desk

Asha’s delivery desk promised a customer their meal by **12:40**. At **12:00**, when the courier leaves, the app needs a message. At dispatch the model uses available information such as address and traffic to score the order. A cutoff turns the score into **On track** or **Likely late**. Either way the courier delivers the order. At arrival we record whether the delivery met its deadline and compare that label with the forecast.

**Route through the notebook** · Run the scoring rule → follow two orders into the confusion matrix → read precision, recall, accuracy, and F1 → compare cutoffs → separate one ROC point from the full AUC.

**Before teaching** · Run all cells once. Start with **Make a forecast from a score** and change the cutoff in the short code cell. The visual outputs are expandable so you can reveal the results one at a time. Each section restates which orders and cutoff it uses.

Positive (**P**, label 1) means the order meets its deadline. Negative (**N**, label 0) means it arrives late. The 20 original synthetic orders provide scores and labels for hand calculations. Higher scores rank an order as more likely to arrive on time. Reading 0.45 as a 45% chance would require a calibration check.

[Open the classification playground](https://sharanharsoor.github.io/ml-learning-lab/classification-regression-metrics/#classification)

In [ ]:
# @title Load the original delivery examples { display-mode: "form" }
from html import escape
from math import isclose
from IPython.display import HTML, display

scores = [0.98, 0.96, 0.94, 0.92, 0.9, 0.85, 0.8, 0.75, 0.7, 0.65, 0.6, 0.55, 0.5, 0.45, 0.4, 0.3, 0.2, 0.15, 0.1, 0.05]
actual = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0]
order_ids = [f'D{i:02d}' for i in range(1, len(actual) + 1)]

## Make a forecast from a score

For our first dispatch policy, show **On track** when the score is at least **0.25**. D14 and D19 are two order IDs from the historical register. At dispatch D14 scores 0.45 and D19 scores 0.10. Run the rule across all 20 orders, then read the two example forecasts. Change the cutoff to 0.10 and compare D19 again. Equality counts as On track.

In [ ]:
cutoff = 0.25
predicted = [int(score >= cutoff) for score in scores]
correct = sum(p == y for p, y in zip(predicted, actual))
accuracy = correct / len(actual)

for i in (13, 18):
    message = 'On track' if predicted[i] else 'Likely late'
    print(f'{order_ids[i]} · score {scores[i]:.2f} · {message}')
print(f'{correct}/{len(actual)} correct forecasts · accuracy {accuracy:.1%}')

## Trace two orders into the confusion matrix

Read each order at dispatch. Then open **Reveal the recorded arrival** to see where it belongs. Rows are actual outcomes and columns are forecast outcomes. P means on time and N means late. TP and TN are correct forecasts. FP means a late order was forecast On track. FN means an on-time order was forecast Likely late. Each order adds one count to its cell. A forecast can change when we move the cutoff. The recorded arrival stays fixed.

In [ ]:
# @title Show selectable order traces { display-mode: "form" }
style = """<style>
.delivery-notebook { padding: 16px; border: 1px solid #cadcd5; border-radius: 12px;
 color: #173430; background: #f1f7f4; font: 16px/1.55 system-ui, sans-serif; }
.delivery-notebook h3, .delivery-notebook p { margin: 0 0 10px; }
.delivery-notebook details { padding: 10px 12px; margin: 7px 0; border: 1px solid #cadcd5;
 border-radius: 8px; background: #fff; }
.delivery-notebook summary { cursor: pointer; font-weight: 700; }
.delivery-notebook details > div { margin-top: 10px; }
.delivery-notebook table { width: 100%; border-collapse: collapse; margin: 8px 0; text-align: left; }
.delivery-notebook :is(th, td) { padding: 7px; border-bottom: 1px solid #cadcd5; }
.delivery-notebook .hit { background: #d5eee5; color: #134e41; font-weight: bold; }
.delivery-notebook .figure { overflow-x: auto; }
.delivery-notebook svg { display: block; width: 100%; max-width: 560px; min-width: 340px; }
</style>"""

def panel(title, html):
    display(HTML(style + f'<section class="delivery-notebook"><h3>{escape(title)}</h3>{html}</section>'))

def disclosure(title, html, group=None, opened=False):
    name = '' if group is None else f' name="{escape(group)}"'
    state = ' open' if opened else ''
    return f'<details{name}{state}><summary>{escape(title)}</summary><div>{html}</div></details>'

def outcome(y, prediction):
    if prediction:
        return 'TP' if y else 'FP'
    return 'FN' if y else 'TN'

def matrix(highlight):
    cells = {kind: f'<td{" class=hit" if kind == highlight else ""}>{kind}</td>'
             for kind in ('TP', 'FN', 'FP', 'TN')}
    return (f'<table><thead><tr><th>Actual outcome</th><th>Forecast P</th><th>Forecast N</th></tr></thead>'
            f'<tbody><tr><th scope="row">Actual P</th>{cells["TP"]}{cells["FN"]}</tr>'
            f'<tr><th scope="row">Actual N</th>{cells["FP"]}{cells["TN"]}</tr></tbody></table>')

def trace_order(i, opener):
    p = predicted[i]
    y = actual[i]
    identifier = order_ids[i]
    message = 'On track' if p else 'Likely late'
    label = 'on time' if y else 'late'
    comparison = '≥' if p else '&lt;'
    prediction_line = f'<p>At dispatch, {scores[i]:.2f} {comparison} {cutoff:.2f}. Display <strong>{message}</strong>.</p>'
    observed = (f'<p>{identifier} actually arrived <strong>{label}</strong> and adds one {outcome(y, p)}.</p>'
                + matrix(outcome(y, p)))
    return disclosure(identifier, prediction_line + disclosure('Reveal the recorded arrival', observed),
                      group='delivery-order', opened=opener)

panel(f'Follow one order at cutoff {cutoff:.2f}',
      '<p>Choose D14 or D19. Reveal the arrival only after reading the dispatch forecast.</p>'
      + trace_order(13, True) + trace_order(18, False))

## Count all 20 orders

Each order contributes one entry to the matrix. Put actual P, N on the rows and forecast P, N on the columns. This gives **[[TP, FN], [FP, TN]]**. At 0.25 the four counts are **[[15, 1], [1, 3]]**.

**Accuracy** asks how many forecasts were correct overall. **Precision** looks down the predicted-P column. **Recall** looks across the actual-P row. At 0.25 the last two are both 15/16. Both groups include the same 15 correct On track forecasts. The precision column adds D14 (score 0.45, forecast On track, actually late). The recall row adds D19 (score 0.10, forecast Likely late, actually on time).

**F1** is the harmonic mean of precision and recall. Written using counts, it is **2 × TP / (2 × TP + FP + FN)**. At 0.25 this gives **2 × 15 / (2 × 15 + 1 + 1) = 30/32**. F1 is pulled toward the smaller rate. For example, precision 1.0 and recall 0.25 give F1 0.4.

In [ ]:
def measure(cutoff):
    counts = dict.fromkeys(('TP', 'FN', 'FP', 'TN'), 0)
    rows = []
    for identifier, score, y in zip(order_ids, scores, actual):
        tag = outcome(y, int(score >= cutoff))
        counts[tag] += 1
        rows.append((identifier, tag))
    tp, fn, fp, tn = (counts[key] for key in ('TP', 'FN', 'FP', 'TN'))
    positive_predictions = tp + fp
    rates = {
        'Accuracy': (tp + tn) / len(actual),
        'Precision': tp / positive_predictions if positive_predictions else None,
        'Recall': tp / (tp + fn),
        'F1': (2 * tp) / (2 * tp + fp + fn),
        'FPR': fp / (fp + tn),
    }
    return counts, rates, rows

counts, rates, rows = measure(0.25)
print('Actual rows P, N · predicted columns P, N')
print([[counts['TP'], counts['FN']], [counts['FP'], counts['TN']]])
for label, value in rates.items():
    print(f'{label} · {"Undefined" if value is None else f"{value:.2%}"}')
assert counts == {'TP': 15, 'FN': 1, 'FP': 1, 'TN': 3}

## Compare two cutoff policies

Keep the dataset and the score ordering fixed. At a lower cutoff, more orders qualify for On track. Check which arrived on time and which arrived late before saying whether the new policy helps.

Compare 0.25 with 0.10 first. Then open the all-On-track policy, 0.00, to expose the majority-class baseline. Finally try 1.01. With no positive forecasts, the precision denominator is zero and the ratio is undefined. Some libraries provide a configurable replacement value for reporting.

In [ ]:
# @title Show cutoff comparisons { display-mode: "form" }
base_counts, base_rates, base_rows = measure(0.25)

def rate_text(value):
    return 'Undefined' if value is None else f'{value:.2%}'

def comparison_card(other, opened=False):
    counts, rates, rows = measure(other)
    changes = [(identifier, scores[i], actual[i], old, new)
               for i, ((identifier, old), (_, new)) in enumerate(zip(base_rows, rows))
               if old != new]
    measure_names = ('TP', 'FN', 'FP', 'TN', 'Precision', 'Recall', 'Accuracy', 'F1')
    table_rows = ''.join(
        f'<tr><th scope="row">{name}</th><td>{base_counts[name] if name in base_counts else rate_text(base_rates[name])}</td>'
        f'<td>{counts[name] if name in counts else rate_text(rates[name])}</td></tr>' for name in measure_names
    )
    shown = ''.join(
        f'<li><strong>{identifier}</strong> · score {score:.2f} · arrived {"on time" if y else "late"} · {old} to {new}</li>'
        for identifier, score, y, old, new in changes[:5]
    )
    change_text = f'<p>{len(changes)} orders change forecast. Their scores and recorded outcomes stay fixed.</p><ul>{shown}</ul>'
    if len(changes) > 5:
        change_text += '<p>Showing five changes.</p>'
    table = (f'<table><thead><tr><th>Measure</th><th>Cutoff 0.25</th><th>Cutoff {other:.2f}</th></tr></thead>'
             f'<tbody>{table_rows}</tbody></table>')
    return disclosure(f'Compare with cutoff {other:.2f}', table + change_text,
                      group='delivery-cutoff', opened=opened)

panel('Same records, different forecasts',
      '<p>The first column is the 0.25 policy. Choose an alternative for the same 20 orders.</p>'
      + ''.join(comparison_card(other, opened=i == 0) for i, other in enumerate((0.10, 0.50, 0.00, 1.01))))

### Decide what matters for the delivery desk

At 0.10, three scores reach the new cutoff. D17 (0.20) and D18 (0.15) arrived late and become false On track forecasts. D19 (0.10) arrived on time and now receives the correct On track forecast. Recall rises from 15/16 to 16/16. Precision falls from 15/16 to 16/19 because the added predicted positives include two late orders. Across a fixed set of scores, lowering the cutoff cannot reduce recall and can move precision either way.

If an incorrect On track forecast costs ten units and an unnecessary Likely late message costs one, this toy cost is 11 at 0.25 and 30 at 0.10. In another problem, missing an actual positive may cost more. Validate the threshold using agreed costs or a requirement such as minimum recall. Keep an independent test set for final evaluation.

## Read one ROC point and the full curve

At a chosen cutoff, the vertical coordinate is **TPR**, which equals recall, and the horizontal coordinate is **FPR**. TPR divides by actual positives. FPR divides by actual negatives. At 0.25 the coordinate is **(1/4, 15/16)**. At 0.10 it is **(3/4, 1)**.

Start with a cutoff above all scores, then lower it through the distinct values. Every newly included positive moves the point up and every newly included negative moves it right. Tied scores cross together. The curve spans the cutoff policies that can be formed from these scores.

In [ ]:
def operating_point(cutoff):
    _, metrics, _ = measure(cutoff)
    return metrics['FPR'], metrics['Recall']

thresholds = [float('inf'), *sorted(set(scores), reverse=True)]
points = [operating_point(value) for value in thresholds]
area = sum((right_x - left_x) * (left_y + right_y) / 2
           for (left_x, left_y), (right_x, right_y) in zip(points, points[1:]))
point = operating_point(0.25)
print(f'At 0.25 · FPR {point[0]:.2%} · TPR {point[1]:.2%}')
print(f'Trapezoid area · {area:.6f}')

In [ ]:
# @title Compare the ROC operating points { display-mode: "form" }
def roc_picture(cutoff):
    fpr, tpr = operating_point(cutoff)
    def x(rate):
        return 75 + 400 * rate
    def y(rate):
        return 255 - 220 * rate

    curve = ' '.join(f'{x(a):.1f},{y(b):.1f}' for a, b in points)
    ticks = ''.join(
        f'<text x="{x(v)}" y="275" text-anchor="middle">{v:g}</text>'
        f'<text x="63" y="{y(v) + 5}" text-anchor="end">{v:g}</text>'
        for v in (0, 0.25, 0.5, 0.75, 1)
    )
    return f"""<div class="figure"><svg viewBox="0 0 510 305" role="img"
      aria-label="ROC at cutoff {cutoff:.2f}. False-positive rate {fpr:.2%}, recall {tpr:.2%}."
      style="font: 13px system-ui; fill: #173430">
      <path d="M75 35V255H475" stroke="#6c8d81" fill="none"/>
      <path d="M75 255L475 35" stroke="#6c8d81" stroke-dasharray="5 4" fill="none"/>
      <polyline points="{curve}" stroke="#087669" stroke-width="3" fill="none"/>
      <circle cx="{x(fpr)}" cy="{y(tpr)}" r="6" fill="#bd604c" stroke="#fff" stroke-width="2"/>
      {ticks}<text x="275" y="301" text-anchor="middle">False-positive rate</text>
      <text transform="translate(19 150) rotate(-90)" text-anchor="middle">Recall (TPR)</text>
      </svg></div><p>Cutoff <strong>{cutoff:.2f}</strong> gives FPR <strong>{fpr:.2%}</strong> and recall <strong>{tpr:.2%}</strong>.</p>"""

panel('Move the point along one ROC curve',
      '<p>The coral dot marks the cutoff. The green curve uses the complete score ordering.</p>'
      + ''.join(disclosure(f'Cutoff {value:.2f}', roc_picture(value), group='delivery-roc', opened=i == 0)
                for i, value in enumerate((0.25, 0.10, 0.50))))

## Explain AUC using pairs

Pair each of the 16 on-time orders with each of the 4 late orders. Give one point when the on-time order has the higher score and half a point for a tie. For example, on-time D19 scored 0.10. Its pair with late D14 (0.45) earns 0. Its pair with late D20 (0.05) earns 1. The code below groups all 64 pairs by the late order and shows how they add to **59/64 = 0.921875**. This equals the ROC area. A different cutoff keeps the same pairs and scores.

**Interpretation** · This is the empirical chance that an on-time order outranks a late order, with half credit for a tie. Accuracy is about decisions at a cutoff. Calibration asks how often a stated probability matches the observed frequency. Those are separate properties.

In [ ]:
positive_scores = [score for score, y in zip(scores, actual) if y]
late_orders = [(identifier, score) for identifier, score, y in zip(order_ids, scores, actual) if not y]
pair_groups = []
for identifier, score in late_orders:
    wins = sum(p > score for p in positive_scores)
    ties = sum(p == score for p in positive_scores)
    pair_groups.append((identifier, score, wins, ties, wins + ties / 2))

pairs = len(positive_scores) * len(late_orders)
credits = sum(group[-1] for group in pair_groups)
auc = credits / pairs
rows = ''.join(
    f'<tr><th scope="row">{identifier}</th><td>{score:.2f}</td><td>{wins}</td><td>{ties}</td><td>{credit:g} / {len(positive_scores)}</td></tr>'
    for identifier, score, wins, ties, credit in pair_groups
)
panel('Where the AUC numerator comes from',
      f'<table><caption>Compare each late order against all {len(positive_scores)} on-time orders</caption>'
      '<thead><tr><th>Late order</th><th>Score</th><th>On-time wins</th><th>Tied scores</th><th>Pair credit</th></tr></thead>'
      f'<tbody>{rows}</tbody></table><p>{credits:g} credits / {pairs} pairs = <strong>{auc:.6f}</strong></p>')
assert pairs == 64 and credits == 59
assert isclose(area, auc, abs_tol=1e-12)

## Put the pieces together

**Score → cutoff → forecast → compare with the later outcome.** One threshold produces a confusion matrix and operating point. Precision and recall ask about distinct groups. ROC-AUC summarizes the rankings over thresholds. Choose a threshold with validation data and the operational cost of the mistakes, then evaluate that choice on an untouched test set.

For a task where the prediction is itself a number of minutes, continue to the [Regression Metrics playground](https://sharanharsoor.github.io/ml-learning-lab/classification-regression-metrics/#regression).